In [ ]:
# ============================== INSTALLS (run once) ==============================
!pip -q install torch torchvision pycocotools opencv-python tqdm matplotlib seaborn
!pip -q install torch torchvision pycocotools opencv-python tqdm matplotlib

# NOTE: If you need segment-anything, upload the repository folder under drive and a weights .pth,
# and set SAM_MODULE_DIR below; or pip install if internet available.
# !pip -q install git+https://github.com/facebookresearch/segment-anything.git

In [ ]:
!rm -rf /content/drive/*
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# ============================== 0) CLEAN START ===============================
# (Run once to remove any older project folder in Drive.)
import shutil, os, json, zipfile, random, numpy as np, cv2, torch, torchvision
from pathlib import Path
from google.colab import drive
from tqdm import tqdm
import matplotlib.pyplot as plt

# ---- fresh slate
PROJECT_ROOT = "/content/drive/MyDrive/tree_project"
shutil.rmtree(PROJECT_ROOT, ignore_errors=True)
os.makedirs(PROJECT_ROOT, exist_ok=True)


In [ ]:
# ============================== 3) CONFIG ====================================
# If you have a ZIP of a COCO-format dataset, point to it here.
# Your dataset must have train/valid/test folders, each with images + _annotations.coco.json
DATASET_ZIP = "/content/Tree Counting.v1i.coco.zip"   # change if you have a zip
BASE_DIR     = "/content/coconut_dataset"                 # local workspace (fast IO)

# Limit training samples for quick experiments (set to None for full set)
MAX_TRAIN_IMAGES = None   # set None for full training set
MAX_VAL_IMAGES   = None    # optional subsetting for validation

# I/O in Drive
DRIVE_CKPTS   = f"{PROJECT_ROOT}/checkpoints"
DRIVE_RESULTS = f"{PROJECT_ROOT}/results"
for p in [DRIVE_CKPTS, DRIVE_RESULTS]:
    os.makedirs(p, exist_ok=True)

In [ ]:
# ============================== 4) DATA PREP =================================
# Unzip only if missing
if not os.path.exists(BASE_DIR) or not os.listdir(BASE_DIR):
    if os.path.exists(DATASET_ZIP):
        with zipfile.ZipFile(DATASET_ZIP, "r") as z:
            z.extractall(BASE_DIR)
    else:
        print("⚠️ Dataset ZIP not found. If your dataset is already extracted, ignore.")

print("Dataset layout preview:")
!ls -R "$BASE_DIR" | sed -n '1,180p'

In [ ]:
# ============================== 5) DATASET (bbox-only) =======================
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

class CocoBBoxDataset(Dataset):
    """
    COCO detection dataset that ignores 'segmentation' and collapses labels to 1 (coconut tree).
    """
    def __init__(self, img_dir, ann_file, transforms=None, take_first_n=None):
        from pycocotools.coco import COCO
        self.img_dir = img_dir
        self.coco = COCO(ann_file)
        self.img_ids = self.coco.getImgIds()
        if take_first_n is not None:
            self.img_ids = self.img_ids[:int(take_first_n)]
        self.transforms = transforms if transforms is not None else T.ToTensor()

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        info = self.coco.loadImgs([img_id])[0]
        img_path = os.path.join(self.img_dir, info['file_name'])

        # Read image RGB
        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            # fallback: create dummy (rare)
            img_rgb = np.zeros((256,256,3), dtype=np.uint8)
        else:
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # To tensor [C,H,W]
        img_t = self.transforms(img_rgb)

        ann_ids = self.coco.getAnnIds(imgIds=[img_id])
        anns = self.coco.loadAnns(ann_ids)
        boxes, labels, areas, iscrowd = [], [], [], []
        for a in anns:
            x, y, w, h = a['bbox']
            if w <= 1 or h <= 1:  # skip tiny/empty boxes
                continue
            boxes.append([x, y, x + w, y + h])
            labels.append(1)                     # single class
            areas.append(a.get('area', w * h))
            iscrowd.append(a.get('iscrowd', 0))

        target = {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32),
            "labels": torch.as_tensor(labels, dtype=torch.int64),
            "image_id": torch.tensor([img_id]),
            "area": torch.as_tensor(areas, dtype=torch.float32),
            "iscrowd": torch.as_tensor(iscrowd, dtype=torch.int64),
            "path": img_path,
        }
        return img_t, target

def collate_fn(batch):
    imgs, tars = list(zip(*batch))
    return list(imgs), list(tars)

# Folders
TRAIN_DIR = f"{BASE_DIR}/train"
VAL_DIR   = f"{BASE_DIR}/valid"
TEST_DIR  = f"{BASE_DIR}/test"

# Datasets / Loaders
train_ds = CocoBBoxDataset(TRAIN_DIR, f"{TRAIN_DIR}/_annotations.coco.json",
                           transforms=T.ToTensor(), take_first_n=MAX_TRAIN_IMAGES)
val_ds   = CocoBBoxDataset(VAL_DIR,   f"{VAL_DIR}/_annotations.coco.json",
                           transforms=T.ToTensor(), take_first_n=MAX_VAL_IMAGES)
test_ds  = CocoBBoxDataset(TEST_DIR,  f"{TEST_DIR}/_annotations.coco.json",
                           transforms=T.ToTensor(), take_first_n=None)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,  num_workers=2, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=2, shuffle=False, num_workers=2, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False, num_workers=2, collate_fn=collate_fn)

print(f"Train/Val/Test sizes: {len(train_ds)} / {len(val_ds)} / {len(test_ds)}")

In [ ]:
# ============================== 6) MODEL =====================================
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def build_fasterrcnn(num_classes=2, pretrained=True,
                     score_thresh=0.55, box_nms_thresh=0.35,
                     rpn_nms_thresh=0.6, detections_per_img=200):
    # Load base model
    model = fasterrcnn_resnet50_fpn(
        weights="COCO_V1" if pretrained else None
    )
    # Replace head
    in_feats = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_feats, num_classes)

    # Tighten thresholds to reduce duplicates
    # (torchvision exposes these attrs on roi_heads in recent versions)
    model.roi_heads.score_thresh = score_thresh           # drop low scores early
    model.roi_heads.nms_thresh   = box_nms_thresh         # per-class NMS
    model.roi_heads.detections_per_img = detections_per_img

    # Also make RPN slightly stricter
    model.rpn.nms_thresh = rpn_nms_thresh

    return model

frcnn = build_fasterrcnn().to(DEVICE)

# ============================== 7) OPTIM & SCHED =============================
LR = 5e-3
params = [p for p in frcnn.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=LR, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.1)

# ============================== 8) CHECKPOINT I/O ============================
CKPT_PATH = f"{DRIVE_CKPTS}/frcnn_latest.pth"
history = {"train_loss": [], "val_loss": [], "val_map": []}

def save_ckpt(ep):
    torch.save({
        "epoch": ep,
        "model_state": frcnn.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "history": history,
    }, CKPT_PATH)

def maybe_resume():
    start = 0
    if os.path.exists(CKPT_PATH):
        ck = torch.load(CKPT_PATH, map_location=DEVICE)
        frcnn.load_state_dict(ck["model_state"])
        optimizer.load_state_dict(ck["optimizer_state"])
        scheduler.load_state_dict(ck["scheduler_state"])
        h = ck.get("history", {})
        for k in history:
            history[k] = h.get(k, history[k])
        start = ck.get("epoch", 0) + 1
        print(f"🔁 Resumed from epoch {start}")
    return start

start_epoch = maybe_resume()


In [ ]:
# ============================== 9) TRAIN / VAL LOOPS =========================
from pycocotools.cocoeval import COCOeval
import torchvision.ops as ops

# Convert model predictions -> COCO detection format (with NMS)
def preds_to_coco_dt(preds, targets, iou_thr=0.5):
    results = []
    for p, t in zip(preds, targets):
        img_id = int(t['image_id'].item())

        boxes  = p.get('boxes',  torch.zeros((0,4))).detach().cpu()
        scores = p.get('scores', torch.zeros((0,))).detach().cpu()
        labels = p.get('labels', torch.zeros((0,))).detach().cpu()

        if len(boxes) == 0:
            continue

        # 🔹 Apply NMS to remove duplicate coconut detections
        keep = ops.nms(boxes, scores, iou_thr)
        boxes, scores, labels = boxes[keep], scores[keep], labels[keep]

        for b, s, l in zip(boxes.numpy(), scores.numpy(), labels.numpy()):
            if int(l) != 1:   # single class (coconut tree)
                continue
            x1, y1, x2, y2 = b.tolist()
            results.append({
                "image_id": img_id,
                "category_id": 1,
                "bbox": [float(x1), float(y1), float(x2-x1), float(y2-y1)],
                "score": float(s)
            })
    return results


# Run COCO evaluation
def coco_eval(coco_gt, results, iou_type='bbox'):
    if len(results) == 0:
        return 0.0
    coco_dt = coco_gt.loadRes(results)
    ev = COCOeval(coco_gt, coco_dt, iouType=iou_type)
    ev.evaluate(); ev.accumulate(); ev.summarize()
    return float(ev.stats[0])  # mAP@[.5:.95]


# One epoch (train/val)
def one_epoch_loss(loader, train=True):
    frcnn.train() if train else frcnn.eval()
    run = 0.0
    for images, targets in tqdm(loader, desc=("Train" if train else "Val"), leave=False):
        images = [img.to(DEVICE) for img in images]
        targets = [{k: (v.to(DEVICE) if torch.is_tensor(v) else v)
                    for k, v in t.items() if k in ("boxes","labels","image_id","area","iscrowd")}
                   for t in targets]

        with torch.set_grad_enabled(train):
            if train:
                loss_dict = frcnn(images, targets)
                loss = sum(loss_dict.values())
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                run += float(loss.item())
            else:
                # Only forward pass in validation
                _ = frcnn(images)
    return run / max(1, len(loader))


# ============================== CHECKPOINT UTILS ==============================
def save_ckpt(epoch):
    ckpt = {
        "epoch": epoch + 1,
        "model_state": frcnn.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "history": history
    }
    torch.save(ckpt, CKPT_PATH)
    print(f"💾 Saved checkpoint -> {CKPT_PATH}")


def load_ckpt(path):
    ckpt = torch.load(path, map_location=DEVICE)
    frcnn.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    start_epoch = ckpt.get("epoch", 0)
    history.update(ckpt.get("history", history))
    print(f"✅ Loaded checkpoint from {path}, starting at epoch {start_epoch}")
    return start_epoch


# ============================== RESUME TRAINING ==============================
CKPT_PATH = "/content/drive/MyDrive/tree_project 1/checkpoints/frcnn_latest.pth"

# Load checkpoint (resume training if exists)
start_epoch = load_ckpt(CKPT_PATH)

EPOCHS = 10   # total epochs you want to run

for epoch in range(start_epoch, EPOCHS):
    print(f"\n=== Epoch {epoch+1}/{EPOCHS} ===")

    # Train + Val Loss
    tr = one_epoch_loss(train_loader, True)
    vl = one_epoch_loss(val_loader, False)
    scheduler.step()

    # COCO mAP evaluation
    frcnn.eval()
    val_results = []
    with torch.no_grad():
        for images, targets in tqdm(val_loader, desc="Val->mAP"):
            images = [img.to(DEVICE) for img in images]
            preds = frcnn(images)
            val_results += preds_to_coco_dt(preds, targets)   # ✅ NMS applied here
    vmap = coco_eval(val_ds.coco, val_results, iou_type='bbox')

    # Save metrics
    history["train_loss"].append(tr)
    history["val_loss"].append(vl)
    history["val_map"].append(vmap)
    print(f"loss: train {tr:.4f} | val {vl:.4f} | mAP@.50:.95 {vmap:.4f}")

    # Save checkpoint
    save_ckpt(epoch)


In [ ]:
import os, cv2
from tqdm import tqdm

TEST_IMG_DIR = "/content/coconut_dataset/test"
SAVE_DIR = "/content/drive/MyDrive/tree_project 1/detections"
os.makedirs(SAVE_DIR, exist_ok=True)

frcnn.eval()
all_results = []

with torch.no_grad():
    for images, targets in tqdm(test_loader, desc="Testing"):
        images = [img.to(DEVICE) for img in images]
        preds = frcnn(images)

        for p, t in zip(preds, targets):
            img_id = int(t['image_id'].item())
            file_name = test_ds.coco.loadImgs(img_id)[0]['file_name']
            img_path = os.path.join(TEST_IMG_DIR, file_name)

            img = cv2.imread(img_path)
            if img is None:
                continue

            boxes  = p.get('boxes',  torch.zeros((0,4))).detach().cpu()
            scores = p.get('scores', torch.zeros((0,))).detach().cpu()
            labels = p.get('labels', torch.zeros((0,))).detach().cpu()

            keep = ops.nms(boxes, scores, 0.5)
            boxes, scores, labels = boxes[keep], scores[keep], labels[keep]

            for b, s, l in zip(boxes.numpy(), scores.numpy(), labels.numpy()):
                if int(l) != 1:   # only coconut class
                    continue
                x1, y1, x2, y2 = map(int, b.tolist())
                cv2.rectangle(img, (x1, y1), (x2, y2), (0,255,0), 2)
                cv2.putText(img, f"{s:.2f}", (x1, y1-5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)

                all_results.append({
                    "image_id": img_id,
                    "category_id": 1,
                    "bbox": [float(x1), float(y1), float(x2-x1), float(y2-y1)],
                    "score": float(s)
                })

            save_path = os.path.join(SAVE_DIR, file_name)
            cv2.imwrite(save_path, img)

print(f"✅ Saved detections in {SAVE_DIR}")


In [ ]:
import os, cv2, torch, json
from tqdm import tqdm
import torchvision.ops as ops

# ============================
# Paths
# ============================
TEST_IMG_DIR = "/content/coconut_dataset/test"
SAVE_DIR = "/content/drive/MyDrive/tree_project 1/detections_final"
JSON_PATH = os.path.join(SAVE_DIR, "predictions.json")
os.makedirs(SAVE_DIR, exist_ok=True)

# Thresholds
CONF_THRESH = 0.6   # filter low-confidence detections
NMS_IOU = 0.4       # non-max suppression IoU

frcnn.eval()
all_results = []

with torch.no_grad():
    for images, targets in tqdm(test_loader, desc="Testing"):
        images = [img.to(DEVICE) for img in images]
        preds = frcnn(images)

        for p, t in zip(preds, targets):
            img_id = int(t['image_id'].item())
            file_name = test_ds.coco.loadImgs(img_id)[0]['file_name']
            img_path = os.path.join(TEST_IMG_DIR, file_name)

            img = cv2.imread(img_path)
            if img is None:
                continue

            # Get predictions
            boxes  = p.get('boxes',  torch.zeros((0,4))).detach().cpu()
            scores = p.get('scores', torch.zeros((0,))).detach().cpu()
            labels = p.get('labels', torch.zeros((0,))).detach().cpu()

            # Apply confidence filter
            keep_conf = scores > CONF_THRESH
            boxes, scores, labels = boxes[keep_conf], scores[keep_conf], labels[keep_conf]

            # Apply NMS
            if len(boxes) > 0:
                keep = ops.nms(boxes, scores, NMS_IOU)
                boxes, scores, labels = boxes[keep], scores[keep], labels[keep]

            det_count = 0

            for b, s, l in zip(boxes.numpy(), scores.numpy(), labels.numpy()):
                if int(l) != 1:   # only coconut class
                    continue
                x1, y1, x2, y2 = map(int, b.tolist())
                det_count += 1

                # Draw RED box
                cv2.rectangle(img, (x1, y1), (x2, y2), (0,0,255), 2)

                # Draw RED label with score
                cv2.putText(img, f"Coconut {s:.2f}", (x1, y1-5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)

                # Save detection info (COCO format)
                all_results.append({
                    "image_id": img_id,
                    "category_id": 1,
                    "bbox": [float(x1), float(y1), float(x2-x1), float(y2-y1)],
                    "score": float(s)
                })

            # Overlay total tree count
            cv2.putText(img, f"Total Trees: {det_count}", (20,40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255,0,0), 3)

            # Save output image
            save_path = os.path.join(SAVE_DIR, file_name)
            cv2.imwrite(save_path, img)

# ============================
# Save predictions JSON
# ============================
with open(JSON_PATH, "w") as f:
    json.dump(all_results, f, indent=4)

print(f"✅ Saved detections in {SAVE_DIR}")
print(f"✅ Predictions JSON saved at {JSON_PATH}")


In [ ]:
import numpy as np
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import json

# ==============================
# Paths
# ==============================
GT_JSON = "/content/coconut_dataset/test/_annotations.coco.json"
PRED_JSON = "/content/drive/MyDrive/tree_project 1/detections_final/predictions.json"

# ==============================
# Load GT + predictions
# ==============================
coco_gt = COCO(GT_JSON)
coco_dt = coco_gt.loadRes(PRED_JSON)

# ==============================
# Part 1: COCO Evaluation
# ==============================
coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

# ==============================
# Part 2: Accuracy-style metrics
# ==============================
TP, FP, FN = 0, 0, 0

for img_id in coco_gt.getImgIds():
    # ground truth boxes
    gt_ann_ids = coco_gt.getAnnIds(imgIds=[img_id])
    gt_anns = coco_gt.loadAnns(gt_ann_ids)
    gt_boxes = [g["bbox"] for g in gt_anns]

    # predicted boxes
    pred_ann_ids = coco_dt.getAnnIds(imgIds=[img_id])
    pred_anns = coco_dt.loadAnns(pred_ann_ids)
    pred_boxes = [p["bbox"] for p in pred_anns]

    matched_gt = set()
    for pb in pred_boxes:
        px, py, pw, ph = pb
        best_iou, best_gt = 0, None
        for i, gb in enumerate(gt_boxes):
            gx, gy, gw, gh = gb
            # IoU calculation
            ix1, iy1 = max(px, gx), max(py, gy)
            ix2, iy2 = min(px+pw, gx+gw), min(py+ph, gy+gh)
            iw, ih = max(0, ix2-ix1), max(0, iy2-iy1)
            inter = iw * ih
            union = pw*ph + gw*gh - inter
            iou = inter / union if union > 0 else 0
            if iou > best_iou:
                best_iou, best_gt = iou, i

        if best_iou >= 0.5 and best_gt not in matched_gt:
            TP += 1
            matched_gt.add(best_gt)
        else:
            FP += 1

    FN += len(gt_boxes) - len(matched_gt)

# Compute metrics
precision = TP / (TP+FP) if (TP+FP) > 0 else 0
recall = TP / (TP+FN) if (TP+FN) > 0 else 0
f1 = 2*precision*recall / (precision+recall) if (precision+recall) > 0 else 0
accuracy = TP / (TP+FP+FN) if (TP+FP+FN) > 0 else 0

print("\n========== Accuracy Metrics ==========")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"TP={TP}, FP={FP}, FN={FN}")
